In [1]:
# =============================================================================
# Diameter and material reconstruction -- Montpellier network
# -----------------------------------------------------------------------------
# Detects and corrects erroneous pipe diameter and material attributes, and
# predicts missing ones, using a chain-based topological scoring framework.
# Pipes are grouped into "chains" (maximal linear sequences of pipes with no
# branching in between); within a chain, each pipe's diameter and material are
# scored against its immediate upstream/downstream neighbours and against the
# empirical diameter-material co-occurrence distribution learned from the
# reference data. The scoring is calibrated by a grid or random search over
# the per-pattern weights.
# =============================================================================
import pandas as pd
import numpy as np
import networkx as nx
from sklearn.metrics import (
    precision_score, recall_score, f1_score,
    confusion_matrix, matthews_corrcoef
)
from itertools import product
from joblib import Parallel, delayed

In [2]:
# Load the preprocessed pipe and node tables for Montpellier
network = "Montpellier"

df_node = pd.read_pickle(f"../data/{network}/df_node_{network}_invert.pkl")
df_pipe = pd.read_pickle(f"../data/{network}/df_pipe_{network}_invert.pkl")

In [3]:
# Build the directed graph of the sewer network. Nodes carry no attribute
# here (only structure); each edge represents one pipe, carrying its
# diameter, identifier, material and 2D length.
G = nx.DiGraph()

for _, node in df_node.iterrows():
    G.add_node(
        node["node_id"],
    )

for _, pipe in df_pipe.iterrows():
    G.add_edge(
        pipe["initial_node"],
        pipe["terminal_node"],
        diameter = pipe["diameter"],
        pipe_id = pipe["pipe_id"],
        material = pipe["material"],
        length = pipe["length"]
    )

In [4]:
# ── Chain extraction ──────────────────────────────────────────────────────
# Group pipes into maximal linear "chains": a chain is a sequence of pipes
# for which every intermediate node has exactly one incoming and one outgoing
# pipe (no branching). Within such a chain, diameter and material are
# expected to remain locally consistent, which is the assumption exploited by
# the scoring functions below. Each pipe is assigned to exactly one chain.
consecutif_pipe = []

for (init, end) in G.edges():
    used = False
    for cons_pipe in consecutif_pipe:
        if (init, end) in cons_pipe:
            used = True
            break
    if used:
        continue
        
    consecutif_pipe.append([])
    consecutif_pipe[-1].append((init, end))
    current = end
    if len(list(G.predecessors(end))) == 1:
        while True:
            successors = list(G.successors(current))
            if len(successors) != 1:
                break
            if len(list(G.predecessors(successors[0]))) != 1:
                consecutif_pipe[-1].append((current, successors[0]))
                break
            consecutif_pipe[-1].append((current, successors[0]))
            current = successors[0]

    current = init
    if len(list(G.successors(init))) == 1:
        while True:
            predecessors = list(G.predecessors(current))
            if len(predecessors) != 1:
                break
            if len(list(G.successors(predecessors[0]))) != 1:
                consecutif_pipe[-1].insert(0, (predecessors[0], current))
                break
            consecutif_pipe[-1].insert(0, (predecessors[0], current))
            current = predecessors[0]

    
        

In [5]:
# Flatten the chain list into a long-format dataframe with one row per pipe.
# Each pipe carries its position within its chain, boundary flags, and its
# neighbouring pipes' attributes (prev_/next_) obtained via pandas shift on
# each chain group. These neighbour columns are the raw inputs of the
# pattern-based scoring in the next cells.
rows = []

for chain_id, chain in enumerate(consecutif_pipe):
    for position, (init, end) in enumerate(chain):
        data = G.edges[(init, end)]
        rows.append({
            "chain_id": chain_id,
            "position": position,
            "chain_length": len(chain),
            "init_node": init,
            "end_node": end,
            "pipe_id": data["pipe_id"],
            "length": data["length"],
            "material": data["material"],
            "diameter": data["diameter"],
            "is_first": True if (position == 0) else False,
            "is_last": True if (position == (len(chain) - 1)) else False,
            "is_boundary": True if (position == 0 or position == (len(chain) - 1)) else False,
            
        })
df_chain = pd.DataFrame(rows)

for col in ["diameter", "material", "pipe_id", "length"]:
    df_chain[f"prev_{col}"] = df_chain.groupby("chain_id")[col].shift(1)
    df_chain[f"next_{col}"] = df_chain.groupby("chain_id")[col].shift(-1)

df_chain.head()

,chain_id,position,chain_length,init_node,end_node,pipe_id,length,material,diameter,is_first,is_last,is_boundary,prev_diameter,next_diameter,prev_material,next_material,prev_pipe_id,next_pipe_id,prev_length,next_length
0,0,0,3,ass_rega_0017886,ass_rega_0022915,ass_cana_0001409,16.145234,Asbestos-cement,150.0,True,False,True,NaN,150.0,NaN,Asbestos-cement,NaN,ass_cana_0004008,NaN,27.177544
1,0,1,3,ass_rega_0022915,ass_rega_0021922,ass_cana_0004008,27.177544,Asbestos-cement,150.0,False,False,False,150.0,150.0,Asbestos-cement,Asbestos-cement,ass_cana_0001409,ass_cana_0018437,16.145234,30.710957
2,0,2,3,ass_rega_0021922,ass_rega_0019265,ass_cana_0018437,30.710957,Asbestos-cement,150.0,False,True,True,150.0,NaN,Asbestos-cement,NaN,ass_cana_0004008,NaN,27.177544,NaN
3,1,0,2,ass_rega_0017914,ass_rega_0025179,ass_cana_0024365,53.475509,Asbestos-cement,150.0,True,False,True,NaN,150.0,NaN,Asbestos-cement,NaN,ass_cana_0019943,NaN,24.608807
4,1,1,2,ass_rega_0025179,ass_rega_0023778,ass_cana_0019943,24.608807,Asbestos-cement,150.0,False,True,True,150.0,NaN,Asbestos-cement,NaN,ass_cana_0024365,NaN,53.475509,NaN


In [6]:
# Empirical diameter/material co-occurrence probability, learned from the
# subset of pipes with both attributes known. Rare diameters (fewer than 20
# occurrences) are dropped to keep the estimate stable. Rows are materials,
# columns are diameters; each row is normalised so that entries encode
# P(diameter | material). This table is later consulted to score the
# plausibility of a given (diameter, material) pair.
def prob_diam_mat(df_chain):
    df_ref = df_chain[
        df_chain["diameter"].notna()
        & df_chain["material"].notna()
        & (df_chain["material"] != "Undetermined")
    ].copy()
    
    counts = df_ref["diameter"].value_counts()
    common_diameters = counts[counts > 20].index
    
    df_ref = df_ref[df_ref["diameter"].isin(common_diameters)]
    
    compat = pd.crosstab(df_ref["material"], df_ref["diameter"])
    compat_prob = compat.div(compat.sum(axis=1), axis=0)
    
    return (compat_prob)

In [7]:
# ── Baseline scoring of the observed (diameter, material) pair ─────────────
# For each pipe, three scores are combined into current_score:
#   * diameter_score: how the observed diameter fits its neighbours in the
#     chain (constant / progressive / isolated / boundary configurations),
#   * material_score: analogous scoring for the material attribute,
#   * material_diameter_score: how plausible the (diameter, material) pair
#     is under the empirical co-occurrence distribution.
# The absolute weight of each pattern (scores dict) is left as a
# hyperparameter, calibrated by grid/random search downstream.

def material_diameter_score(diameter, material, compat_prob, scores):
    """Score a (diameter, material) pair against the empirical co-occurrence
    distribution. Rare or unseen pairs receive a low (typically negative)
    score, frequent pairs a high positive one."""
    if pd.isna(diameter):
        return 0
    if pd.isna(material) or material == "Undetermined":
        return 0
    if diameter not in compat_prob.columns:
        return scores["pct0"]
    p = compat_prob.loc[material, diameter]
    if p >= 0.25:   return scores["pct25"]
    elif p >= 0.10: return scores["pct10"]
    elif p >= 0.03: return scores["pct3"]
    else:
        return scores["pct0"]

def apply_score(df_chain, compat_prob, scores):
    """Compute diameter_score, material_score, material_diameter_score and
    the summed current_score for every pipe. Uses vectorised numpy.select
    over the mutually exclusive neighbourhood patterns."""

    prev = df_chain["prev_diameter"]
    curr = df_chain["diameter"]
    nxt  = df_chain["next_diameter"]
    is_boundary = df_chain["is_boundary"]
    is_first    = df_chain["is_first"]
    is_last     = df_chain["is_last"]

    # ── Interior pipes (both neighbours belong to the same chain) ─────────
    mask_no_information  = ~is_boundary & prev.isna() & nxt.isna() & curr.isna()
    mask_not_enough      = ~is_boundary & prev.isna() & nxt.isna() & curr.notna()
    mask_nan_bet_same    = ~is_boundary & prev.notna() & (prev == nxt) & curr.isna()
    mask_nan_uncertain   = ~is_boundary & prev.notna() & nxt.notna() & (prev != nxt) & curr.isna()
    mask_constant        = ~is_boundary & (prev == nxt) & (prev == curr)
    mask_isolated        = ~is_boundary & (prev == nxt) & (curr != nxt)
    mask_same_af_be      = ~is_boundary & (prev != nxt) & ((prev == curr) | (nxt == curr)) & curr.notna()
    mask_prog_increase   = ~is_boundary & (prev < curr) & (curr < nxt)
    mask_prog_decrease   = ~is_boundary & (prev > curr) & (curr > nxt)

    # ── Chain head (only a downstream neighbour is available) ─────────────
    mask_first_no_information = is_first & nxt.isna() & curr.isna()
    mask_first_not_enough = is_first & nxt.isna() & curr.notna()
    mask_first_constant   = is_first & nxt.notna() & (nxt == curr)
    mask_first_bound_diff = is_first & nxt.notna() & (curr != nxt)

    # ── Chain tail (only an upstream neighbour is available) ──────────────
    mask_last_no_information = is_last & prev.isna() & curr.isna()
    mask_last_not_enough  = is_last & prev.isna() & curr.notna()
    mask_last_constant    = is_last & prev.notna() & (prev == curr)
    mask_last_bound_diff  = is_last & prev.notna() & (curr != prev)

    df_chain["diameter_score"] = np.select(
        [
            mask_no_information,
            mask_not_enough,
            mask_nan_bet_same,
            mask_nan_uncertain,
            mask_constant,
            mask_isolated,
            mask_same_af_be,
            mask_prog_increase,
            mask_prog_decrease,
            mask_first_no_information,
            mask_first_not_enough,
            mask_first_constant,
            mask_first_bound_diff,
            mask_last_no_information,
            mask_last_not_enough,
            mask_last_constant,
            mask_last_bound_diff,
        ],
        [
            scores["no_information_diameter"],
            scores["not_enough_information_diameter"],
            scores["nan_between_same"],
            scores["nan_uncertain"],
            scores["constant"],
            scores["isolated_diameter"],
            scores["same_diameter_after_or_before"],
            scores["progressive_increase"],
            scores["progressive_decreasing"],
            scores["no_information_diameter"],
            scores["not_enough_information_diameter"],
            scores["same_diameter_after_or_before"],
            scores["boundary_different_diameter"],
            scores["no_information_diameter"],
            scores["not_enough_information_diameter"],
            scores["same_diameter_after_or_before"],
            scores["boundary_different_diameter"],
        ],
        default=scores["unknown"]
    )

    # ── Material scoring: same logic, but "Undetermined" is treated as NaN ─
    prev = df_chain["prev_material"]
    curr = df_chain["material"]
    nxt  = df_chain["next_material"]
    is_boundary = df_chain["is_boundary"]
    is_first    = df_chain["is_first"]
    is_last     = df_chain["is_last"]

    unk_prev = prev.isna() | (prev == "Undetermined")
    unk_curr = curr.isna() | (curr == "Undetermined")
    unk_nxt  = nxt.isna()  | (nxt  == "Undetermined")

    # ── Interior ──────────────────────────────────────────────────────────
    mask_no_information = ~is_boundary & unk_prev & unk_nxt & unk_curr
    mask_not_enough     = ~is_boundary & unk_prev & unk_nxt & ~unk_curr
    mask_stable         = ~is_boundary & (prev == nxt) & (prev == curr) & ~unk_prev
    mask_isolated       = ~is_boundary & (prev == nxt) & (prev != curr) & ~unk_curr & ~unk_prev
    mask_unk_bet_same   = ~is_boundary & (prev == nxt) & unk_curr & ~unk_prev
    mask_unk_uncertain  = ~is_boundary & (prev != nxt) & unk_curr & ~unk_prev & ~unk_nxt
    mask_same_after     = ~is_boundary & (prev != nxt) & ((prev == curr) | (nxt == curr)) & ~unk_curr

    # ── Chain head ────────────────────────────────────────────────────────
    mask_first_no_information = is_first & unk_nxt & unk_curr
    mask_first_not_enough = is_first & unk_nxt
    mask_first_stable     = is_first & ~unk_nxt & (nxt == curr)
    mask_first_boundary   = is_first & ~unk_nxt & (nxt != curr)

    # ── Chain tail ────────────────────────────────────────────────────────
    mask_last_no_information = is_last & unk_prev & unk_curr
    mask_last_not_enough  = is_last & unk_prev
    mask_last_stable      = is_last & ~unk_prev & (prev == curr)
    mask_last_boundary    = is_last & ~unk_prev & (prev != curr)

    df_chain["material_score"] = np.select(
        [
            mask_no_information,
            mask_not_enough,
            mask_stable,
            mask_isolated,
            mask_unk_bet_same,
            mask_unk_uncertain,
            mask_same_after,
            mask_first_no_information,
            mask_first_not_enough,
            mask_first_stable,
            mask_first_boundary,
            mask_last_no_information,
            mask_last_not_enough,
            mask_last_stable,
            mask_last_boundary,
        ],
        [
            scores["no_information_material"],
            scores["not_enough_information_material"],
            scores["stable"],
            scores["isolated_material"],
            scores["unknown_between_same"],
            scores["unknown_uncertain"],
            scores["same_material_after_or_before"],
            scores["no_information_material"],
            scores["not_enough_information_material"],
            scores["same_material_after_or_before"],
            scores["boundary_different_material"],
            scores["no_information_material"],
            scores["not_enough_information_material"],
            scores["same_material_after_or_before"],
            scores["boundary_different_material"],
        ],
        default=scores["unknown"]
    )

    # ── Diameter/material co-occurrence score, from the empirical table ───
    materials = list(compat_prob.index)
    diameters = list(compat_prob.columns)
    mat_dia_scores = {}
    for d in diameters:
        for m in materials:
            mat_dia_scores[(d, m)] = material_diameter_score(d, m, compat_prob, scores)

    df_chain["material_diameter_score"] = pd.MultiIndex.from_arrays([df_chain["diameter"], df_chain["material"]]).map(mat_dia_scores).fillna(scores["pct0"])

    df_chain["current_score"] = (
        df_chain["diameter_score"]
        + df_chain["material_score"]
        + df_chain["material_diameter_score"]
    )

    return df_chain


In [8]:
# ── Candidate scoring: score every possible (diameter, material) value ────
# For each candidate value d (resp. m) drawn from the reference vocabulary,
# compute the score the pipe would receive if its attribute were replaced by
# d (resp. m), leaving its neighbours untouched. This yields per-pipe columns
# dia_score{d} and mat_score{m} used by the correction/prediction stage to
# pick the best replacement. Boundary rules mirror those of apply_score.

def apply_diameter_score(df_chain, scores, diameters):
    for d in diameters:
        prev = df_chain["prev_diameter"]
        nxt  = df_chain["next_diameter"]
        is_boundary = df_chain["is_boundary"]
        is_first    = df_chain["is_first"]
        is_last     = df_chain["is_last"]

        # ── Interior pipes ─────────────────────────────────────────────
        mask_not_enough      = ~is_boundary & prev.isna() & nxt.isna()
        mask_constant        = ~is_boundary & (prev == nxt) & (prev == d)
        mask_isolated        = ~is_boundary & (prev == nxt) & (d != nxt)
        mask_same_af_be      = ~is_boundary & (prev != nxt) & ((prev == d) | (nxt == d))
        mask_prog_increase   = ~is_boundary & (prev < d) & (d < nxt)
        mask_prog_decrease   = ~is_boundary & (prev > d) & (d > nxt)

        # ── Chain head ─────────────────────────────────────────────────
        mask_first_not_enough = is_first & nxt.isna()
        mask_first_constant   = is_first & nxt.notna() & (nxt == d)
        mask_first_bound_diff = is_first & nxt.notna() & (d != nxt)

        # ── Chain tail ─────────────────────────────────────────────────
        mask_last_not_enough  = is_last & prev.isna()
        mask_last_constant    = is_last & prev.notna() & (prev == d)
        mask_last_bound_diff  = is_last & prev.notna() & (d != prev)

        df_chain[f"dia_score{d}"] = np.select(
            [
                mask_not_enough,
                mask_constant,
                mask_isolated,
                mask_same_af_be,
                mask_prog_increase,
                mask_prog_decrease,
                mask_first_not_enough,
                mask_first_constant,
                mask_first_bound_diff,
                mask_last_not_enough,
                mask_last_constant,
                mask_last_bound_diff,
            ],
            [
                scores["not_enough_information_diameter"],
                scores["constant"],
                scores["isolated_diameter"],
                scores["same_diameter_after_or_before"],
                scores["progressive_increase"],
                scores["progressive_decreasing"],
                scores["not_enough_information_diameter"],
                scores["same_diameter_after_or_before"],
                scores["boundary_different_diameter"],
                scores["not_enough_information_diameter"],
                scores["same_diameter_after_or_before"],
                scores["boundary_different_diameter"],
            ],
            default=scores["unknown"]
        )
    return df_chain

def apply_material_score(df_chain, scores, materials):
    for m in materials:
        prev = df_chain["prev_material"]
        nxt  = df_chain["next_material"]
        is_boundary = df_chain["is_boundary"]
        is_first    = df_chain["is_first"]
        is_last     = df_chain["is_last"]

        unk_prev = prev.isna() | (prev == "Undetermined")
        unk_nxt  = nxt.isna()  | (nxt  == "Undetermined")

        # ── Interior pipes ─────────────────────────────────────────────
        mask_not_enough     = ~is_boundary & unk_prev & unk_nxt
        mask_stable         = ~is_boundary & (prev == nxt) & (prev == m)
        mask_isolated       = ~is_boundary & (prev == nxt) & (prev != m)
        mask_same_after     = ~is_boundary & (prev != nxt) & ((prev == m) | (nxt == m))

        # ── Chain head ─────────────────────────────────────────────────
        mask_first_not_enough = is_first & unk_nxt
        mask_first_stable     = is_first & ~unk_nxt & (nxt == m)
        mask_first_boundary   = is_first & ~unk_nxt & (nxt != m)

        # ── Chain tail ─────────────────────────────────────────────────
        mask_last_not_enough  = is_last & unk_prev
        mask_last_stable      = is_last & ~unk_prev & (prev == m)
        mask_last_boundary    = is_last & ~unk_prev & (prev != m)

        df_chain[f"mat_score{m}"] = np.select(
            [
                mask_not_enough,
                mask_stable,
                mask_isolated,
                mask_same_after,
                mask_first_not_enough,
                mask_first_stable,
                mask_first_boundary,
                mask_last_not_enough,
                mask_last_stable,
                mask_last_boundary,
            ],
            [
                scores["not_enough_information_material"],
                scores["stable"],
                scores["isolated_material"],
                scores["same_material_after_or_before"],
                scores["not_enough_information_material"],
                scores["same_material_after_or_before"],
                scores["boundary_different_material"],
                scores["not_enough_information_material"],
                scores["same_material_after_or_before"],
                scores["boundary_different_material"],
            ],
            default=scores["unknown"]
        )
    return df_chain


In [9]:
# Reference vocabularies: only diameters/materials present in the empirical
# co-occurrence table are considered as candidates in the reconstruction step.
compat_prob = prob_diam_mat(df_chain)
materials = list(compat_prob.index)
diameters = list(compat_prob.columns)

print(materials)
print(diameters)


['Asbestos-cement', 'Concrete', 'Glass', 'Metal', 'Plastic']
[63.0, 75.0, 90.0, 100.0, 110.0, 125.0, 150.0, 160.0, 180.0, 200.0, 250.0, 300.0, 315.0, 350.0, 400.0, 450.0, 500.0, 600.0, 700.0, 800.0, 1000.0, 1200.0, 1300.0, 1400.0, 1500.0, 2000.0, 2600.0]


In [10]:
# ── Injected-anomaly protocol: correction setting ─────────────────────────
# A fraction `pourcent` of pipes with a known attribute is randomly picked;
# the true value is replaced by another value drawn uniformly from the
# vocabulary. Neighbour columns (prev_/next_) are regenerated once after all
# injections so that a pipe adjacent to a modified one sees the corrupted
# value. Ground truth (pipe_id, true value) is returned for later scoring.
def inject_anomalies_diameter(df_chain, compat_prob, pourcent = 0.1):
    X = df_chain.copy()
    modified_idx = []
    X["modif_diameter"] = 0
    nb_error = 0
    diameters = list(compat_prob.columns)

    while(nb_error < pourcent * len(X)):
        id = rng.integers(0, len(X))
        if X.loc[id, "modif_diameter"] == 0 and pd.notna(X.loc[id, "diameter"]):
            error = rng.choice(diameters)
            if X.loc[id, "diameter"] != error:
                modified_idx.append(id)
                X.loc[id, "modif_diameter"] = 1
                X.loc[id, "diameter"] = error
                nb_error += 1

    # Regenerate neighbour columns once after all injections.
    X["prev_diameter"] = X.groupby("chain_id")["diameter"].shift(1)
    X["next_diameter"] = X.groupby("chain_id")["diameter"].shift(-1)

    y_diameter = df_chain.loc[modified_idx, ["pipe_id", "diameter"]].copy()
    return X, y_diameter

def inject_anomalies_material(df_chain, compat_prob, pourcent = 0.1):
    X = df_chain.copy()
    modified_idx = []
    X["modif_material"] = 0
    nb_error = 0
    materials = list(compat_prob.index)

    while(nb_error < pourcent * len(X)):
        id = rng.integers(0, len(X))
        if X.loc[id, "modif_material"] == 0 and not (pd.isna(X.loc[id, "material"]) or X.loc[id, "material"] == "Undetermined"):
            error = rng.choice(materials)
            if X.loc[id, "material"] != error:
                modified_idx.append(id)
                X.loc[id, "modif_material"] = 1
                X.loc[id, "material"] = error
                nb_error += 1
    # Regenerate neighbour columns once after all injections.
    X["prev_material"] = X.groupby("chain_id")["material"].shift(1)
    X["next_material"] = X.groupby("chain_id")["material"].shift(-1)

    y_material = df_chain.loc[modified_idx, ["pipe_id", "material"]].copy()

    return X, y_material


In [11]:
# ── Injected-mask protocol: prediction setting ────────────────────────────
# A fraction `pourcent` of pipes with a known attribute is randomly picked
# and its attribute is set to NaN (diameter) or "Undetermined" (material).
# The reconstruction is then evaluated on its ability to recover the hidden
# true values, on the same set of pipes.
def mask_diameter(df_chain, compat_prob, pourcent = 0.1):
    X = df_chain.copy()
    modified_idx = []
    X["modif_diameter"] = 0
    nb_error = 0
    diameters = list(compat_prob.columns)

    while(nb_error < pourcent * len(X)):
        id = rng.integers(0, len(X))
        if X.loc[id, "modif_diameter"] == 0 and pd.notna(X.loc[id, "diameter"]):
            error = np.nan
            modified_idx.append(id)
            X.loc[id, "modif_diameter"] = 1
            X.loc[id, "diameter"] = error
            nb_error += 1

    # Regenerate neighbour columns once after all injections.
    X["prev_diameter"] = X.groupby("chain_id")["diameter"].shift(1)
    X["next_diameter"] = X.groupby("chain_id")["diameter"].shift(-1)

    y_diameter = df_chain.loc[modified_idx, ["pipe_id", "diameter"]].copy()
    return X, y_diameter

def mask_material(df_chain, compat_prob, pourcent = 0.1):
    X = df_chain.copy()
    modified_idx = []
    X["modif_material"] = 0
    nb_error = 0
    materials = list(compat_prob.index)

    while(nb_error < pourcent * len(X)):
        id = rng.integers(0, len(X))
        if X.loc[id, "modif_material"] == 0 and not (pd.isna(X.loc[id, "material"]) or X.loc[id, "material"] == "Undetermined"):
            error = "Undetermined"
            modified_idx.append(id)
            X.loc[id, "modif_material"] = 1
            X.loc[id, "material"] = error
            nb_error += 1

    # Regenerate neighbour columns once after all injections.
    X["prev_material"] = X.groupby("chain_id")["material"].shift(1)
    X["next_material"] = X.groupby("chain_id")["material"].shift(-1)

    y_material = df_chain.loc[modified_idx, ["pipe_id", "material"]].copy()

    return X, y_material


In [12]:
# ── Joint correction/prediction of diameter and material ──────────────────
# For every pipe, evaluate the score of every (candidate diameter, candidate
# material) pair as the sum of:
#   * the candidate diameter's neighbourhood score (dia_score{d}),
#   * the candidate material's neighbourhood score (mat_score{m}),
#   * the co-occurrence score of the pair (from the empirical table),
#   * minus a penalty term that discourages changing the observed value
#     (single change: one penalty; both attributes changed: doubled).
# The (d, m) pair with the highest score is retained; the pipe is corrected
# only if that best score strictly exceeds the score of the current pair.
# This threshold prevents the algorithm from replacing values by ties or by
# marginal improvements.
def correct_predict_diameter_material(df_chain, compat_prob, scores, verbose = True):
    mat_dia_scores = {}
    materials = list(compat_prob.index)
    diameters = list(compat_prob.columns)
    old_score = df_chain["current_score"]
    for diameter in diameters:
        for material in materials:
            mat_dia_scores[(diameter, material)] = material_diameter_score(diameter, material, compat_prob, scores)

    df_chain = apply_diameter_score(df_chain, scores, diameters)
    df_chain = apply_material_score(df_chain, scores, materials)

    # Score tensor: axis 0 = pipes, axis 1 = candidate diameters,
    # axis 2 = candidate materials.
    score = np.zeros((len(df_chain), len(list(compat_prob.columns)), len(list(compat_prob.index))))
    for d in range(len(diameters)):
        for m in range(len(materials)):
            diameter_same = (df_chain["diameter"] == diameters[d])
            material_same = (df_chain["material"] == materials[m])

            # Change penalty: 0 if the candidate matches the observed value,
            # single penalty if one of the two attributes changes, doubled if
            # both change.
            penalty = np.where(
                diameter_same & material_same, 0,
                np.where(diameter_same | material_same, scores["penality"], 2 * scores["penality"])
            )

            score[:, d, m] = (
                df_chain[f"dia_score{diameters[d]}"] +
                df_chain[f"mat_score{materials[m]}"] +
                mat_dia_scores[(diameters[d], materials[m])] -
                penalty
            )

    # Find the argmax over (diameter, material) for every pipe.
    score_1d = score.reshape(len(df_chain) , -1)
    best_score_idx = (np.unravel_index(np.argmax(score_1d, axis = 1), (len(diameters), len(materials))))
    # Replace only if the best candidate strictly improves on the current one.
    df_chain["corrected_diameter"] = np.where(score[np.arange(len(df_chain)), best_score_idx[0], best_score_idx[1]] > old_score,
                                              np.array(diameters)[best_score_idx[0]],
                                              df_chain["diameter"]
                                             )
    df_chain["corrected_material"] = np.where(score[np.arange(len(df_chain)), best_score_idx[0], best_score_idx[1]] > old_score,
                                              np.array(materials)[best_score_idx[1]],
                                              df_chain["material"]
                                             )
    return df_chain


In [13]:
# ── Evaluation: correction setting ─────────────────────────────────────────
# A pipe is a positive if its attribute was actually modified by the
# injection protocol; it is a predicted positive if the algorithm changed the
# observed value. We report MCC (primary criterion, since positives are much
# rarer than negatives), precision, recall, F1, and the exact-correction
# rate on true positives (proportion of TP for which the recovered value
# equals the ground truth). Metrics are also split between pipes belonging
# to a chain (chain_length > 1) and isolated pipes, since the scoring relies
# heavily on neighbour agreement.
def evaluateReconstruction_diameter_material(X, y_diameter=None, y_material=None,
                                              scores=None, feature="diameter", pourcent=0.1, verbose=True):
    if feature == "diameter":
        mask_known = pd.notna(X["diameter"])
        X = X.loc[mask_known]
        y_true = X["modif_diameter"]
        y_ref = y_diameter
        col_corrected = "corrected_diameter"
        col_original = "diameter"
        y_pred = (X[col_corrected] != X[col_original])

    elif feature == "material":
        mask_known = X["material"] != "Undetermined"
        X = X.loc[mask_known]
        y_true = X["modif_material"]
        y_ref = y_material
        col_corrected = "corrected_material"
        col_original = "material"
        y_pred = (X[col_corrected] != X[col_original])

    elif feature == "both":
        mask_known = ((pd.notna(X["diameter"])) | (X["material"] != "Undetermined"))
        X = X.loc[mask_known]
        # A pipe counts as modified if either attribute was changed by the injection.
        y_true = ((X["modif_diameter"] == 1) | (X["modif_material"] == 1))
        # A pipe counts as detected if either attribute was corrected.
        y_pred = (
            (pd.notna(X["diameter"]) & (X["corrected_diameter"] != X["diameter"])) |
            ((X["material"] != "Undetermined") & (X["corrected_material"] != X["material"]))
        )
        col_corrected = None
        col_original  = None


    # ── Global metrics ─────────────────────────────────────────────
    mcc_global  = matthews_corrcoef(y_true, y_pred)
    prec_global = precision_score(y_true, y_pred)
    rec_global  = recall_score(y_true, y_pred)
    f1_global   = f1_score(y_true, y_pred)

    # ── Chain vs. isolated pipes ───────────────────────────────────
    X_chain    = X[X["chain_length"] > 1]
    X_isolated = X[X["chain_length"] == 1]

    if feature == "both":
        y_t_chain = ((X_chain["modif_diameter"] == 1) | (X_chain["modif_material"] == 1))
        y_t_iso   = ((X_isolated["modif_diameter"] == 1) | (X_isolated["modif_material"] == 1))
        y_p_chain = (
            (X_chain["corrected_diameter"] != X_chain["diameter"]) |
            (X_chain["corrected_material"] != X_chain["material"])
        )
        y_p_iso = (
            (X_isolated["corrected_diameter"] != X_isolated["diameter"]) |
            (X_isolated["corrected_material"] != X_isolated["material"])
        )
    else:
        y_t_chain = X_chain[f"modif_{feature}"]
        y_t_iso   = X_isolated[f"modif_{feature}"]
        y_p_chain = (X_chain[col_corrected] != X_chain[col_original])
        y_p_iso   = (X_isolated[col_corrected] != X_isolated[col_original])


    mcc_chain  = matthews_corrcoef(y_t_chain, y_p_chain)
    prec_chain = precision_score(y_t_chain, y_p_chain)
    rec_chain  = recall_score(y_t_chain, y_p_chain)
    mcc_iso    = matthews_corrcoef(y_t_iso, y_p_iso)
    prec_iso   = precision_score(y_t_iso, y_p_iso)
    rec_iso    = recall_score(y_t_iso, y_p_iso)

    # ── Correction quality on true positives ───────────────────────
    # A TP is exactly correct only if the recovered value equals the ground truth.
    tp_idx = X[(y_true == 1) & (y_pred == 1)].index

    if feature == "both":
        # Exact correction = both diameter AND material recovered.
        tp_df = X.loc[tp_idx, ["pipe_id", "diameter", "corrected_diameter",
                                "material", "corrected_material"]].copy()
        tp_df = tp_df.merge(
            y_diameter.rename(columns={"diameter": "true_diameter"}), on="pipe_id", how="left"
        ).merge(
            y_material.rename(columns={"material": "true_material"}), on="pipe_id", how="left"
        )
        # If only one attribute was injected on a pipe, the other true value
        # is the currently observed one.
        tp_df["true_diameter"] = tp_df["true_diameter"].fillna(
            tp_df["pipe_id"].map(dict(zip(X["pipe_id"], X["diameter"])))
        )
        tp_df["true_material"] = tp_df["true_material"].fillna(
            tp_df["pipe_id"].map(dict(zip(X["pipe_id"], X["material"])))
        )
        tp_df["exact_correction"] = (
            (tp_df["corrected_diameter"] == tp_df["true_diameter"]) &
            (tp_df["corrected_material"] == tp_df["true_material"])
        )
    else:
        tp_df = X.loc[tp_idx, ["pipe_id", col_original, col_corrected]].copy()
        tp_df = tp_df.merge(
            y_ref.rename(columns={col_original: "true_value"}), on="pipe_id", how="left"
        )
        tp_df["exact_correction"] = tp_df[col_corrected] == tp_df["true_value"]

    exact_correction_rate = tp_df["exact_correction"].mean()

    # ── False positive / false negative counts ────────────────────
    fp_idx  = X[(y_true == 0) & (y_pred == 1)].index
    fn_idx  = X[(y_true == 1) & (y_pred == 0)].index
    fp_rate = len(fp_idx) / max((y_true == 0).sum(), 1)
    fn_rate = len(fn_idx) / max((y_true == 1).sum(), 1)

    # ── Corrected / modified ratio on chains ──────────────────────
    if feature == "both":
        n_corrected_chain = (
            (X_chain["corrected_diameter"] != X_chain["diameter"]) |
            (X_chain["corrected_material"]  != X_chain["material"])
        ).sum()
        n_modified_chain  = ((X_chain["modif_diameter"] == 1) | (X_chain["modif_material"] == 1)).sum()
    else:
        n_corrected_chain = (X_chain[col_corrected] != X_chain[col_original]).sum()
        n_modified_chain  = X_chain[f"modif_{feature}"].sum()
    ratio_chain = n_corrected_chain / max(n_modified_chain, 1)

    if verbose:
        print(f"Feature : {feature} | Pourcent : {pourcent:.0%}")
        print(f"── Global ──")
        print(f"MCC       : {mcc_global:.3f}")
        print(f"Precision : {prec_global:.3f}")
        print(f"Recall    : {rec_global:.3f}")
        print(f"F1        : {f1_global:.3f}")
        print(f"Exact correction rate : {exact_correction_rate:.3f}")
        print(f"── Chains ──")
        print(f"MCC : {mcc_chain:.3f} | Precision : {prec_chain:.3f} | Recall : {rec_chain:.3f}")
        print(f"── Isolated ──")
        print(f"MCC : {mcc_iso:.3f} | Precision : {prec_iso:.3f} | Recall : {rec_iso:.3f}")
        print("Confusion matrix :")
        print(confusion_matrix(y_true, y_pred))

    result = {
        **(scores),
        "feature":               feature,
        "pourcent":              pourcent,
        "mcc":                   float(mcc_global),
        "precision":             float(prec_global),
        "recall":                float(rec_global),
        "f1":                    float(f1_global),
        "exact_correction_rate": float(exact_correction_rate),
        "fp_rate":               float(fp_rate),
        "fn_rate":               float(fn_rate),
        "mcc_chain":             float(mcc_chain),
        "precision_chain":       float(prec_chain),
        "recall_chain":          float(rec_chain),
        "ratio_chain":           float(ratio_chain),
        "mcc_isolated":          float(mcc_iso),
        "precision_isolated":    float(prec_iso),
        "recall_isolated":       float(rec_iso),
        "n_modified":            int(y_true.sum()),
        "n_detected":            int(y_pred.sum()),
        "n_tp":                  len(tp_idx),
        "n_fp":                  len(fp_idx),
        "n_fn":                  len(fn_idx),
        "n_isolated":            len(X_isolated),
        "n_chain":               len(X_chain),
    }
    return result


In [14]:
# ── Evaluation: prediction setting ─────────────────────────────────────────
# In the prediction protocol, evaluation is restricted to the set of pipes
# whose value was hidden by the mask protocol. `pct_predicted` is the
# fraction of hidden values for which the algorithm returned a non-missing
# reconstruction, and `exact_correction_rate` is the fraction of those
# predictions that exactly recover the ground-truth value. Classical
# precision/recall are not reported here since predictions are, by
# construction, restricted to the hidden set.
def evaluatePrediction_diameter_material(X, y_diameter=None, y_material=None,
                                              scores=None, feature="diameter", pourcent=0.1, verbose=True):
    if feature == "diameter":
        y_true = X["modif_diameter"]
        y_ref = y_diameter
        col_corrected = "corrected_diameter"
        col_original  = "diameter"
        y_pred = (X["corrected_diameter"].notna()) & y_true

    elif feature == "material":
        y_true = X["modif_material"]
        y_ref = y_material
        col_corrected = "corrected_material"
        col_original  = "material"
        y_pred = (X["corrected_material"] != "Undetermined") & y_true

    elif feature == "both":
        y_true = ((X["modif_diameter"] == 1) | (X["modif_material"] == 1))
        y_pred = (((X["corrected_diameter"].notna()) | (X["corrected_material"] != "Undetermined")) & y_true)
        col_corrected = None
        col_original  = None

    pct_predicted = float(y_pred.sum() / y_true.sum())

    # ── Correction quality on true positives ───────────────────────
    tp_idx = X[(y_true == 1) & (y_pred == 1)].index

    if feature == "both":
        # Exact correction = both diameter AND material recovered.
        tp_df = X.loc[tp_idx, ["pipe_id", "diameter", "corrected_diameter",
                                "material", "corrected_material"]].copy()
        tp_df = tp_df.merge(
            y_diameter.rename(columns={"diameter": "true_diameter"}), on="pipe_id", how="left"
        ).merge(
            y_material.rename(columns={"material": "true_material"}), on="pipe_id", how="left"
        )
        tp_df["true_diameter"] = tp_df["true_diameter"].fillna(
            tp_df["pipe_id"].map(dict(zip(X["pipe_id"], X["diameter"])))
        )
        tp_df["true_material"] = tp_df["true_material"].fillna(
            tp_df["pipe_id"].map(dict(zip(X["pipe_id"], X["material"])))
        )
        tp_df["exact_correction"] = (
            (tp_df["corrected_diameter"] == tp_df["true_diameter"]) &
            (tp_df["corrected_material"] == tp_df["true_material"])
        )
    else:
        tp_df = X.loc[tp_idx, ["pipe_id", col_original, col_corrected]].copy()
        tp_df = tp_df.merge(
            y_ref.rename(columns={col_original: "true_value"}), on="pipe_id", how="left"
        )
        tp_df["exact_correction"] = tp_df[col_corrected] == tp_df["true_value"]

    exact_correction_rate = tp_df["exact_correction"].mean()

    if verbose:
        print(f"Feature : {feature} | Pourcent : {pourcent:.0%}")
        print(f"── Global ──")
        print(f"Exact correction rate : {exact_correction_rate:.3f}")
        print(f"% predicted NaN : {pct_predicted:.1%}")

    result = {
        **(scores if scores is not None else {}),
        "feature":               feature,
        "pourcent":              pourcent,
        "exact_correction_rate": float(exact_correction_rate),
        "pct_predicted":         float(pct_predicted),
        "n_modified":            int(y_true.sum()),
        "n_detected":            int(y_pred.sum()),
    }
    return result


In [15]:
# ── Random search over the pattern weights ────────────────────────────────
# Sample N score configurations uniformly from per-parameter grids. The sign
# constraints below encode the qualitative interpretation of each pattern:
# coherent patterns (constant, progressive, high co-occurrence) must yield
# positive scores; incoherent ones (isolated, boundary mismatch, rare
# co-occurrence) must yield negative scores.
rng = np.random.default_rng(42)
N = 5000  # number of configurations to sample

variable_params = {
    # Must be positive -- penalises modifying an observed value.
    "penality":                      list(range(1, 8)),

    # Coherent patterns -> positive scores.
    "constant":                      list(range(0, 6)),
    "same_diameter_after_or_before": list(range(0, 6)),
    "pct25":                         list(range(0, 6)),
    "pct10":                         list(range(0, 6)),
    "pct3":                          list(range(0, 6)),
    "stable":                        list(range(0, 6)),
    "same_material_after_or_before": list(range(0, 6)),
    "progressive_increase":          list(range(0, 6)),
    "progressive_decreasing":        list(range(0, 6)),
    "not_enough_information_diameter": list(range(0, 6)),
    "not_enough_information_material": list(range(0, 6)),
    "no_information_diameter":       list(range(0, 6)),
    "no_information_material":       list(range(0, 6)),

    # Incoherent patterns -> negative scores.
    "pct0":                         list(range(-5, 1)),
    "isolated_diameter":            list(range(-5, 1)),
    "boundary_different_diameter":  list(range(-5, 1)),
    "isolated_material":            list(range(-5, 1)),
    "boundary_different_material":  list(range(-5, 1)),
    "unknown_between_same":         list(range(-5, 1)),
    "nan_between_same":             list(range(-5, 1)),

    # Ambiguous patterns -> small symmetric range around zero.
    "nan_uncertain":                 list(range(-3, 3)),
    "unknown":                       list(range(-3, 3)),
    "unknown_uncertain":             list(range(-3, 3)),
}

jobs = []
for _ in range(N):
    scores = {}
    for key, values in variable_params.items():
        scores[key] = int(rng.choice(values))
    jobs.append((scores))

print(f"Number of jobs : {len(jobs)}")


Number of jobs : 5000


In [16]:
# ── Focused grid search over the pattern weights ──────────────────────────
# After the random search identifies the parameters that most correlate with
# the target metric, fix the less influential ones (fixed_scores) and enumerate
# a small dense grid on the remaining parameters (variable_params).

fixed_scores = {
    # Few neighbours known -> mildly positive, so as not to force a change.
    "not_enough_information_diameter": 1,
    "not_enough_information_material": 1,
    "no_information_diameter":         2,
    "no_information_material":         2,

    # Weakly informative patterns.
    "nan_uncertain":                   0,
    "unknown_between_same":            -3,
    "unknown":                         0,
    "unknown_uncertain":               0,
    "progressive_decreasing":          0,
    "progressive_increase":            0,
    "pct3":                            0,
    "isolated_diameter":               -2,
    "isolated_material":               -2,
    "pct10":                           2,
}

variable_params = {
    "penality":                        [3, 6, 9],
    "pct25":                           [3, 6],
    "same_material_after_or_before":   [1, 3],
    "same_diameter_after_or_before":   [1, 3],
    "stable":                          [3, 6],
    "constant":                        [3, 6],
    "pct0":                            [-4, -2],
    "boundary_different_material":     [-2, 0],
    "boundary_different_diameter":     [-2, 0],
    "nan_between_same":            [-5, -2],
}

jobs = []
keys = list(variable_params.keys())
for combo in product(*variable_params.values()):
    scores = fixed_scores.copy()
    scores.update(dict(zip(keys, combo)))
    jobs.append((scores))

print(f"Number of jobs : {len(jobs)}")


Number of jobs : 1536


In [17]:
# ── Single-configuration runner ────────────────────────────────────────────
# Wrap one score configuration in a try/except so that one failing evaluation
# does not abort the whole parallel sweep. Returns the evaluation summary of
# the corresponding correction or prediction protocol.
def run_one(scores, df_chain, pourcent, feature, y_diameter = None, y_material = None, corr_pred = "correction"):
    results = []
    X = df_chain.copy()
    try:
        X = apply_score(X, compat_prob, scores)
        X = correct_predict_diameter_material(X, compat_prob, scores, verbose = False)

        if corr_pred == "correction":
            summary = evaluateReconstruction_diameter_material(X, y_diameter, y_material,
                                              scores=scores, feature= feature, pourcent=pourcent, verbose=False)
        elif corr_pred == "prediction":
            summary = evaluatePrediction_diameter_material(X, y_diameter, y_material,
                                              scores=scores, feature= feature, pourcent=pourcent, verbose=False)
        results.append(summary)
    except Exception as e:
        print(f"ERROR : {e}")
    return results


In [21]:
# ── Full sweep over injected-anomaly (or masking) rates and score configs ──
# For each `pourcent` value: draw one anomaly injection (or mask) with a
# fixed seed for reproducibility, then evaluate every score configuration in
# parallel on that shared X. All per-configuration results are concatenated
# into a single results dataframe.
compat_prob = prob_diam_mat(df_chain)
pourcent = [0.01, 0.05, 0.1, 0.3, 0.5, 0.8]
feature = "both"
corr_pred = "correction"
all_results_combined = []

for pct in pourcent:
    rng = np.random.default_rng(42)
    df_modified = df_chain.copy()
    y_diameter = None
    y_material = None

    if corr_pred == "correction":
        if feature == "diameter" or feature == "both":
            df_modified, y_diameter = inject_anomalies_diameter(df_modified, compat_prob, pourcent = pct)
        if feature == "material" or feature == "both":
            df_modified, y_material = inject_anomalies_material(df_modified, compat_prob, pourcent = pct)
    if corr_pred == "prediction":
        if feature == "diameter" or feature == "both":
            df_modified, y_diameter = mask_diameter(df_modified, compat_prob, pourcent = pct)
        if feature == "material" or feature == "both":
            df_modified, y_material = mask_material(df_modified, compat_prob, pourcent = pct)

    all_results = Parallel(n_jobs=16, verbose=10)(
        delayed(run_one)(scores, df_modified, pct, feature, y_diameter = y_diameter, y_material = y_material, corr_pred = corr_pred)
        for scores in jobs
    )
    all_results_combined.extend(all_results)

results = [r for sublist in all_results_combined for r in sublist]
df_results = pd.DataFrame(results)
print(f"Saved : {len(df_results)} rows")

In [23]:
# ── Post-hoc parameter selection ──────────────────────────────────────────
# Rank the varied parameters by the correlation with the final score, to
# identify which ones drive performance and should be kept in the next
# (finer) grid search. In the correction setting we optimise the MCC; in the
# prediction setting we combine coverage (pct_predicted) and accuracy
# (exact_correction_rate) with equal weight.
param_cols = list(variable_params.keys())
if corr_pred == "prediction":
    df_results["score"] = (
        0.5 * df_results["pct_predicted"] +
        0.5 * df_results["exact_correction_rate"]
    )
elif corr_pred == "correction":
    df_results["score"] = df_results["mcc"]


correlations = df_results[param_cols].corrwith(df_results["score"]).sort_values(ascending=False)
print(correlations)

In [ ]:
# NOTE: replace XXXXXXXX with the run date, and swap "gridSearch" for
# "randomSearch" depending on which sweep was executed above.
df_results.to_csv(f"../results_dia_mat/{feature}Reconstruction_{network}_gridSearch_XXXXXXXX.csv", index=False)